In [0]:
%pip install requests

In [0]:
import json
import re
import uuid
import hashlib
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
)
##test

In [0]:
RUN_ID = str(uuid.uuid4())
FETCHED_AT = datetime.now(timezone.utc)

MIN_RATE = 125

ROLE_KEYWORDS = [
    "software developer",
    "software engineer",
    "senior developer",
    "senior software engineer",
    "full stack",
    "full-stack",
    "technical lead",
    "tech lead",
    "solution architect",
    "payments architect",
    ".net",
    "aws",
    "backend",
    "back end",
]

JOB_DOMAINS = [
    "seek.co.nz",
    "www.seek.co.nz",
    "nz.seek.com",
    "trademe.co.nz",
    "www.trademe.co.nz",
    "nz.indeed.com",
    "indeed.com",
    "jobs.govt.nz",
]

SEARCH_SOURCES = [
    {
        "source": "bing_seek",
        "type": "bing_rss",
        "allowed_domains": ["seek.co.nz", "www.seek.co.nz", "nz.seek.com"],
        "queries": [
            'site:seek.co.nz/job Auckland contract "software developer"',
            'site:seek.co.nz/job Auckland contract "software engineer"',
            'site:seek.co.nz/job Auckland contract "senior developer"',
            'site:seek.co.nz/job Auckland contract "senior software engineer"',
            'site:seek.co.nz/job Auckland contract "full stack"',
            'site:seek.co.nz/job Auckland contract "technical lead"',
            'site:seek.co.nz/job Auckland contract "solution architect"',
            'site:seek.co.nz/job Auckland contract "payments architect"',
            'site:seek.co.nz/job Auckland contract ".NET"',
            'site:seek.co.nz/job Auckland contract "AWS"',
            'site:seek.co.nz/job Auckland "$125"',
            'site:seek.co.nz/job Auckland "$130"',
            'site:seek.co.nz/job Auckland "$140"',
        ],
    },
    {
        "source": "bing_trademe",
        "type": "bing_rss",
        "allowed_domains": ["trademe.co.nz", "www.trademe.co.nz"],
        "queries": [
            'site:trademe.co.nz/a/jobs Auckland contract "software developer"',
            'site:trademe.co.nz/a/jobs Auckland contract "software engineer"',
            'site:trademe.co.nz/a/jobs Auckland contract "senior developer"',
            'site:trademe.co.nz/a/jobs Auckland contract "technical lead"',
            'site:trademe.co.nz/a/jobs Auckland contract "solution architect"',
            'site:trademe.co.nz/a/jobs Auckland "$125"',
            'site:trademe.co.nz/a/jobs Auckland "$130"',
        ],
    },
    {
        "source": "bing_indeed",
        "type": "bing_rss",
        "allowed_domains": ["nz.indeed.com", "indeed.com"],
        "queries": [
            'site:nz.indeed.com Auckland contract "software developer"',
            'site:nz.indeed.com Auckland contract "software engineer"',
            'site:nz.indeed.com Auckland contract "senior developer"',
            'site:nz.indeed.com Auckland contract "technical lead"',
        ],
    },
    {
        "source": "bing_jobs_govt_nz",
        "type": "bing_rss",
        "allowed_domains": ["jobs.govt.nz"],
        "queries": [
            'site:jobs.govt.nz Auckland contract "software developer"',
            'site:jobs.govt.nz Auckland contract "software engineer"',
            'site:jobs.govt.nz Auckland contract "solution architect"',
            'site:jobs.govt.nz Auckland contract "technical lead"',
        ],
    },
]

CUSTOM_RSS_FEEDS = []

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS job_watch")

spark.sql("""
CREATE TABLE IF NOT EXISTS job_watch.bronze_search_runs (
  run_id STRING,
  source STRING,
  fetched_at TIMESTAMP,
  query STRING,
  response_json STRING
)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS job_watch.silver_seek_results (
  source STRING,
  result_id STRING,
  title STRING,
  url STRING,
  content STRING,
  hourly_min DOUBLE,
  hourly_max DOUBLE,
  first_seen_at TIMESTAMP,
  last_seen_at TIMESTAMP
)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS job_watch.gold_seek_high_rate_roles (
  source STRING,
  result_id STRING,
  title STRING,
  url STRING,
  content STRING,
  hourly_min DOUBLE,
  hourly_max DOUBLE,
  last_seen_at TIMESTAMP
)
""")

In [0]:
def parse_hourly_rate(text: str):
    """
    Extract hourly rates from text like:
    - $125/h
    - $125 - $130/h
    - $130–$135 + GST
    - $150 per hour
    """

    if not text:
        return None, None

    t = text.lower()
    t = t.replace(",", "")
    t = t.replace("nzd", "$")
    t = t.replace("per hour", "/h")
    t = t.replace("per hr", "/h")
    t = t.replace("hourly", "/h")
    t = t.replace("p/h", "/h")
    t = t.replace("ph", "/h")
    t = t.replace("–", "-")
    t = t.replace("—", "-")

    hourly_hint = any(
        hint in t
        for hint in [
            "/h",
            "hour",
            "+ gst",
            "contract rate",
            "hourly rate",
        ]
    )

    if not hourly_hint:
        return None, None

    nums = re.findall(r"\$?\s*(\d{2,3}(?:\.\d+)?)", t)
    nums = [float(n) for n in nums]

    # Filter out random numbers like dates, job IDs, years, etc.
    nums = [n for n in nums if 50 <= n <= 300]

    if not nums:
        return None, None

    if len(nums) == 1:
        return nums[0], nums[0]

    first_two = nums[:2]
    return min(first_two), max(first_two)

In [0]:
def make_result_id(url: str, title: str):
    value = f"{url}|{title}".strip().lower()
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

In [0]:
import xml.etree.ElementTree as ET
from urllib.parse import (
    urlencode,
    urlparse,
    urlunparse,
    parse_qsl,
    urlencode as url_encode,
)


def canonicalize_url(url: str) -> str:
    if not url:
        return ""

    parsed = urlparse(url)
    query_pairs = parse_qsl(parsed.query, keep_blank_values=True)

    blocked_prefixes = ("utm_",)
    blocked_exact = {
        "tracking",
        "srsltid",
        "gclid",
        "fbclid",
    }

    clean_pairs = [
        (k, v)
        for k, v in query_pairs
        if not k.lower().startswith(blocked_prefixes)
        and k.lower() not in blocked_exact
    ]

    clean_query = url_encode(clean_pairs)

    return urlunparse((
        parsed.scheme,
        parsed.netloc.lower(),
        parsed.path.rstrip("/"),
        "",
        clean_query,
        "",
    ))


def parse_rss_or_atom(xml_bytes: bytes):
    root = ET.fromstring(xml_bytes)
    results = []

    for item in root.findall(".//item"):
        title = item.findtext("title") or ""
        link = item.findtext("link") or ""
        description = item.findtext("description") or ""
        pub_date = item.findtext("pubDate") or ""

        results.append({
            "title": title,
            "url": link,
            "content": description,
            "published": pub_date,
        })

    return results


def fetch_rss_url(feed_url: str, source_name: str, query: str = ""):
    headers = {
        "User-Agent": "Mozilla/5.0 job-watch-rss-monitor/1.0",
        "Accept": "application/rss+xml, application/xml, text/xml, */*",
    }

    response = requests.get(
        feed_url,
        headers=headers,
        timeout=30,
    )

    if response.status_code >= 400:
        raise RuntimeError(
            f"RSS request failed. "
            f"source={source_name}, "
            f"status={response.status_code}, "
            f"query={query}, "
            f"url={feed_url}, "
            f"body={response.text[:1000]}"
        )

    text_start = response.text[:300].lower()

    if "<rss" not in text_start and "<?xml" not in text_start:
        raise RuntimeError(
            f"RSS source did not return XML. "
            f"source={source_name}, "
            f"query={query}, "
            f"url={feed_url}, "
            f"content_type={response.headers.get('content-type')}, "
            f"body_start={response.text[:500]}"
        )

    try:
        results = parse_rss_or_atom(response.content)
    except Exception as e:
        raise RuntimeError(
            f"Could not parse RSS XML. "
            f"source={source_name}, "
            f"query={query}, "
            f"url={feed_url}, "
            f"error={str(e)}, "
            f"body_start={response.text[:500]}"
        )

    return {
        "source": source_name,
        "query": query,
        "feed_url": feed_url,
        "results": results,
    }


def bing_rss_search(query: str, source_name: str):
    base_url = "https://www.bing.com/search"

    params = {
        "q": query,
        "format": "rss",
        "cc": "NZ",
        "setmkt": "en-NZ",
    }

    feed_url = f"{base_url}?{urlencode(params)}"
    return fetch_rss_url(feed_url, source_name, query)


def source_search(source_config: dict, query: str):
    source_type = source_config["type"]
    source_name = source_config["source"]

    if source_type == "bing_rss":
        return bing_rss_search(query, source_name)

    raise ValueError(f"Unsupported source type: {source_type}")


def url_allowed(url: str, allowed_domains: list[str]):
    if not allowed_domains:
        return True

    url_lower = url.lower()
    return any(domain.lower() in url_lower for domain in allowed_domains)


def is_probably_job_result(source_name: str, url: str, title: str, content: str):
    text = f"{title} {content} {url}".lower()
    url_lower = url.lower()

    blocked_words = [
        "nz herald",
        "nzherald",
        "rnz",
        "nucamp",
        "nzctu",
        "explainer",
        "employment relations amendment",
        "highest paying",
        "top 10",
        "news",
        "article",
        "blog",
        "bill 2025",
    ]

    if any(word in text for word in blocked_words):
        return False, "blocked article/news keyword"

    if "auckland" not in text and "all-auckland" not in text:
        return False, "missing auckland"

    role_hit = any(keyword in text for keyword in ROLE_KEYWORDS)

    if not role_hit:
        return False, "missing software/dev role keyword"

    if source_name == "bing_seek":
        seek_url_hit = (
            "/job/" in url_lower
            or "-jobs/in-" in url_lower
            or "/jobs/in-" in url_lower
        )

        if not seek_url_hit:
            return False, "seek url does not look like job/search result"

    if source_name == "bing_trademe":
        if "trademe.co.nz" not in url_lower:
            return False, "trademe url does not look valid"

    if source_name == "bing_indeed":
        if "indeed.com" not in url_lower:
            return False, "indeed url does not look valid"

    if source_name == "bing_jobs_govt_nz":
        if "jobs.govt.nz" not in url_lower:
            return False, "jobs.govt.nz url does not look valid"

    return True, "ok"

In [0]:
bronze_rows = []
silver_rows = []
errors = []

seen_canonical_urls = set()

for source_config in SEARCH_SOURCES:
    source_name = source_config["source"]
    allowed_domains = source_config.get("allowed_domains", [])
    queries = source_config.get("queries", [])

    for query in queries:
        print("=" * 100)
        print("SOURCE:", source_name)
        print("QUERY:", query)

        try:
            payload = source_search(source_config, query)

            result_count = len(payload.get("results", []))
            print("RESULTS:", result_count)
            print("FEED URL:", payload.get("feed_url"))

            bronze_rows.append({
                "run_id": RUN_ID,
                "source": source_name,
                "fetched_at": FETCHED_AT.isoformat(),
                "query": query,
                "response_json": json.dumps(payload),
            })

            for item in payload.get("results", []):
                title = item.get("title") or ""
                url = item.get("url") or ""
                content = item.get("content") or ""

                clean_url = canonicalize_url(url)

                print("TITLE:", title)
                print("URL:", clean_url)
                print("CONTENT:", content[:300])
                print()

                if not clean_url:
                    continue

                if not url_allowed(clean_url, allowed_domains):
                    continue

                # Deduplicate across Bing/Google/custom feeds by URL.
                if clean_url in seen_canonical_urls:
                    continue

                seen_canonical_urls.add(clean_url)

                combined_text = f"{title}\n{content}\n{clean_url}"
                hourly_min, hourly_max = parse_hourly_rate(combined_text)

                result_id = make_result_id(clean_url, title)

                silver_rows.append({
                    "source": source_name,
                    "result_id": result_id,
                    "title": title,
                    "url": clean_url,
                    "content": content,
                    "hourly_min": hourly_min,
                    "hourly_max": hourly_max,
                    "first_seen_at": FETCHED_AT.isoformat(),
                    "last_seen_at": FETCHED_AT.isoformat(),
                })

        except Exception as e:
            error_message = f"Failed source={source_name}, query={query}\n{str(e)}"
            print(error_message)
            errors.append(error_message)


# Custom direct RSS feeds, if configured
for feed in CUSTOM_RSS_FEEDS:
    source_name = feed["source"]
    feed_url = feed["url"]
    allowed_domains = feed.get("allowed_domains", [])

    print("=" * 100)
    print("SOURCE:", source_name)
    print("CUSTOM RSS:", feed_url)

    try:
        payload = fetch_rss_url(feed_url, source_name, query=feed_url)

        print("RESULTS:", len(payload.get("results", [])))

        bronze_rows.append({
            "run_id": RUN_ID,
            "source": source_name,
            "fetched_at": FETCHED_AT.isoformat(),
            "query": feed_url,
            "response_json": json.dumps(payload),
        })

        for item in payload.get("results", []):
            title = item.get("title") or ""
            url = item.get("url") or ""
            content = item.get("content") or ""

            clean_url = canonicalize_url(url)

            if not clean_url:
                continue

            if not url_allowed(clean_url, allowed_domains):
                continue
            if not is_probably_job_ad(source_name, clean_url, title, content):
                continue
            if clean_url in seen_canonical_urls:
                continue

            seen_canonical_urls.add(clean_url)

            combined_text = f"{title}\n{content}\n{clean_url}"
            hourly_min, hourly_max = parse_hourly_rate(combined_text)

            result_id = make_result_id(clean_url, title)

            silver_rows.append({
                "source": source_name,
                "result_id": result_id,
                "title": title,
                "url": clean_url,
                "content": content,
                "hourly_min": hourly_min,
                "hourly_max": hourly_max,
                "first_seen_at": FETCHED_AT.isoformat(),
                "last_seen_at": FETCHED_AT.isoformat(),
            })

    except Exception as e:
        error_message = f"Failed custom RSS source={source_name}, url={feed_url}\n{str(e)}"
        print(error_message)
        errors.append(error_message)


print("=" * 100)
print("Bronze rows:", len(bronze_rows))
print("Silver rows:", len(silver_rows))
print("Errors:", len(errors))

if errors:
    raise RuntimeError(
        "One or more sources failed:\n\n" + "\n\n".join(errors)
    )

if len(bronze_rows) == 0:
    raise RuntimeError("No bronze rows returned. No source returned usable responses.")

if len(silver_rows) == 0:
    raise RuntimeError("No silver rows returned. Sources worked, but no URLs passed filters.")

In [0]:
bronze_schema = StructType([
    StructField("run_id", StringType()),
    StructField("source", StringType()),
    StructField("fetched_at", StringType()),
    StructField("query", StringType()),
    StructField("response_json", StringType()),
])

bronze_df = (
    spark.createDataFrame(bronze_rows, bronze_schema)
    .withColumn("fetched_at", F.to_timestamp("fetched_at"))
)

bronze_df.write.mode("append").saveAsTable("job_watch.bronze_search_runs")

In [0]:
silver_schema = StructType([
    StructField("source", StringType()),
    StructField("result_id", StringType()),
    StructField("title", StringType()),
    StructField("url", StringType()),
    StructField("content", StringType()),
    StructField("hourly_min", DoubleType()),
    StructField("hourly_max", DoubleType()),
    StructField("first_seen_at", StringType()),
    StructField("last_seen_at", StringType()),
])

silver_df = (
    spark.createDataFrame(silver_rows, silver_schema)
    .dropDuplicates(["source", "result_id"])
    .withColumn("first_seen_at", F.to_timestamp("first_seen_at"))
    .withColumn("last_seen_at", F.to_timestamp("last_seen_at"))
)

silver_df.createOrReplaceTempView("new_seek_results")

spark.sql("""
MERGE INTO job_watch.silver_seek_results target
USING new_seek_results source
ON target.source = source.source
AND target.result_id = source.result_id

WHEN MATCHED THEN UPDATE SET
  target.title = source.title,
  target.url = source.url,
  target.content = source.content,
  target.hourly_min = source.hourly_min,
  target.hourly_max = source.hourly_max,
  target.last_seen_at = source.last_seen_at

WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE job_watch.gold_seek_high_rate_roles AS
SELECT
  source,
  result_id,
  title,
  url,
  content,
  hourly_min,
  hourly_max,
  last_seen_at
FROM job_watch.silver_seek_results
WHERE hourly_max >= {MIN_RATE}
ORDER BY hourly_max DESC, last_seen_at DESC
""")

In [0]:
display(spark.sql("""
SELECT
  title,
  hourly_min,
  hourly_max,
  url,
  content,
  last_seen_at
FROM job_watch.gold_seek_high_rate_roles
ORDER BY hourly_max DESC, last_seen_at DESC
"""))

In [0]:
display(spark.sql("""
SELECT
  source,
  COUNT(*) AS total_rows,
  COUNT(hourly_max) AS rows_with_rate,
  MAX(hourly_max) AS max_rate
FROM job_watch.silver_seek_results
GROUP BY source
ORDER BY total_rows DESC
"""))

In [0]:
display(spark.sql("""
SELECT
  source,
  title,
  url,
  content,
  hourly_min,
  hourly_max,
  last_seen_at
FROM job_watch.silver_seek_results
ORDER BY last_seen_at DESC
LIMIT 100
"""))